In [2]:
# Importing everything we'll need for Part 1 in one place.
import csv
import json
import numpy as np
import pandas as pd
import requests

# Upload appointment.csv, clinics.csv and followup_notes.csv from your computer.
# (Or use the folder icon on the left of Colab and drag the files in.)
from google.colab import files
uploaded = files.upload()

# Change this if your file is named differently.
FILE = "appointments.csv"

Saving appointment.csv to appointment.csv


In [3]:
# Task 1.1: read with the csv module only (no pandas here)

def parse_row(row):
    """Turn the text values in one CSV row into proper numbers.
    If a value is blank or not a number, int()/float() will raise ValueError,
    and the loop below will catch it."""
    converted = dict(row)  # copy so we don't mess with the original row
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []

with open(FILE, newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)

    for row in reader:
        try:
            accepted.append(parse_row(row))
        except ValueError:
            # Couldn't convert something (e.g. blank age), so park the row here.
            rejected.append(row)

print("Total rows read:", len(accepted) + len(rejected))
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
# Expected: 1020 total, 988 accepted, 32 rejected
# (12 blank age + 20 blank distance_km). Ages like 250 are still accepted,
# because a type check is not a validity check. We handle those in Task 1.3.

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020
Accepted: 988
Rejected: 32


In [4]:
# Task 1.2: start again from the ORIGINAL file, this time with pandas.
# (The accepted/rejected lists from 1.1 are NOT used here.)

df = pd.read_csv(FILE)

print("Shape:", df.shape)          # expect (1020, 14)
print()
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())           # expect age 12, distance_km 20, status 4, appt_date 1
print()
print("Duplicate rows:", df.duplicated().sum())   # expect 20

# (a) drop exact duplicate rows
df = df.drop_duplicates()
print("\nAfter dropping duplicates:", len(df))     # expect 1000

# (b) tidy booking_channel (stray spaces and odd capitalisation)
print("Raw spellings:", df["booking_channel"].nunique())   # expect 16
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Clean spellings:", df["booking_channel"].nunique())  # expect 3 (App, Phone, Walk-In)
print("After cleaning channel:", len(df))

# (c) convert dates; errors="coerce" turns unreadable dates into NaT
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After date conversion:", len(df))

# (d) drop rows where appt_date or status is missing
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing appt_date/status:", len(df))   # expect 990

Shape: (1020, 14)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values per column:
appointment_id      0
patient_id          0
c

In [5]:
# Task 1.4: new features, then NumPy stats with no loops

# lead_days = how many days before the appointment the booking was made
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# our target for Part 3: 1 if the patient didn't show, else 0
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())  # expect 1 to 45
print("No-shows:", df["no_show"].sum(),
      f"({df['no_show'].mean() * 100:.1f}%)")             # expect 218 (22.0%)

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) mean of each column
print("Column means:", X.mean(axis=0).round(2))           # expect about [48.17, 9.50, 635.71, 11.82]

# (b) bookings made more than 14 days ahead (boolean mask)
mask = X[:, 3] > 14
print("lead_days > 14:", mask.sum(), f"({mask.mean() * 100:.1f}%)")   # expect 296 (29.9%)

# (c) z-scores of distance (np.std default = population std, as the brief says)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2:", (np.abs(z) > 2).sum())        # expect 53

lead_days range: 1 to 45
No-shows: 218 (22.0%)
Column means: [   nan    nan 649.7   11.82]
lead_days > 14: 296 (29.9%)
Rows with |z| > 2: 0


In [7]:
# Task 1.5: get the midnight temperature for each clinic from the free API.
# Never invent a number: if the call fails twice, temp_c stays None.

clinics = pd.read_csv("clinics.csv")

def get_temperature(lat, lon):
    """Returns (temperature, reason). Temperature is None if the call failed."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": lon,
              "hourly": "temperature_2m", "forecast_days": 1}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code != 200:
            return None, f"status code {resp.status_code}"
        data = resp.json()
        if "hourly" not in data:
            return None, "'hourly' missing in reply"
        # first value = midnight (UTC) reading, our stand-in for the day's temperature
        return data["hourly"]["temperature_2m"][0], "ok"
    except requests.exceptions.RequestException as e:
        return None, f"request error: {e}"

temps = []
for _, row in clinics.iterrows():
    temp, reason = get_temperature(row["latitude"], row["longitude"])
    if temp is None:
        # one retry, as the brief asks
        temp, reason = get_temperature(row["latitude"], row["longitude"])
    if temp is None:
        print(f"FAILED: {row['clinic_name']} -> {reason}")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)
# Expect 8 rows and temp_c between -10 and 50 (the exact values change daily).

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.8  
1    18.2  
2    18.2  
3    16.0  
4    16.0  
5    25.2  
6    17.0  
7    15.3  


In [8]:
# Task 1.6: final export + load_data.sql

# Sort, then turn both date columns into plain YYYY-MM-DD text
final_cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
              "age", "gender", "distance_km", "fee_inr", "booking_channel",
              "visit_type", "reminder_sent", "prior_no_shows", "status",
              "lead_days", "no_show"]

out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[final_cols]

# Sanity checks before exporting
print("appointment_id unique:", out["appointment_id"].is_unique)         # expect True
print("Missing values in any column:", out.isnull().sum().sum())          # expect 0
print("Shape:", out.shape)                                                # expect (990, 16)
out.to_csv("appointments_clean.csv", index=False)

# Turn one Python value into SQL text:
# None/NaN -> NULL (unquoted), text -> 'quoted', numbers -> as they are.
def sql_value(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NULL"
    if isinstance(v, str):
        return "'" + v.replace("'", "''") + "'"   # no apostrophes here, but harmless
    return str(v)

clinics_final = pd.read_csv("clinics_enriched.csv")
clinics_final = clinics_final.astype(object).where(clinics_final.notnull(), None)

with open("load_data.sql", "w") as f:
    # clinics FIRST, since appointments has a foreign key pointing at them
    for rec in clinics_final.to_dict("records"):
        values = ", ".join(sql_value(rec[c]) for c in clinics_final.columns)
        f.write(f"INSERT INTO clinics VALUES ({values});\n")

    for rec in out.to_dict("records"):
        values = ", ".join(sql_value(rec[c]) for c in final_cols)
        f.write(f"INSERT INTO appointments VALUES ({values});\n")

with open("load_data.sql") as f:
    print("INSERT statements:", sum(1 for _ in f))   # expect 998 (8 + 990)

# Download everything for your GitHub repo
for name in ["appointments_clean.csv", "clinics_enriched.csv",
             "load_data.sql", "rejected_rows.json"]:
    files.download(name)

appointment_id unique: True
Missing values in any column: 32
Shape: (990, 16)
INSERT statements: 998


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# Task 2.2 (Python side): groupby should match the SQL counts exactly
appts = pd.read_csv("appointments_clean.csv")
clin = pd.read_csv("clinics_enriched.csv")

merged = appts.merge(clin, on="clinic_id")
check = (merged.groupby(["clinic_name", "city"])
               .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
               .reset_index())
check["no_show_pct"] = (100 * check["no_shows"] / check["appointments"]).round(1)

print(check.sort_values("no_show_pct", ascending=False).to_string(index=False))

NameError: name 'pd' is not defined

In [ ]:
# Task 2.2 (Python side): reproduce the SQL clinic counts with groupby
# df is the cleaned table from Part 1, clinics_final is clinics_enriched.csv

appts = pd.read_csv("appointments_clean.csv")
clin = pd.read_csv("clinics_enriched.csv")

merged = appts.merge(clin, on="clinic_id")

check = (merged.groupby(["clinic_name", "city"])
               .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
               .reset_index())
check["no_show_pct"] = (100 * check["no_shows"] / check["appointments"]).round(1)

# All 8 clinics, same order as the SQL query without HAVING
print(check.sort_values("no_show_pct", ascending=False).to_string(index=False))

# Same list with the >= 95 appointments filter (like the HAVING version)
print()
print(check[check["appointments"] >= 95]
      .sort_values("no_show_pct", ascending=False)
      .to_string(index=False))